# QG testbed — realism scenarios: when does 4DVar lose its advantage?

`qg2l_pipeline.ipynb` is a favourable case for 4DVar: same equations / parameters / forcing for the truth and the
DA model (truncation-only model error), observations without representativeness error, exactly known R, B estimated
from the truth, perfect H, periodic domain, unlimited optimisation budget. This notebook switches these
assumptions off one at a time (and all together), and compares for each scenario

* **4DVarNet** (retrained on each new dataset; **3 training seeds** in `baseline` and `realistic`),
* **4DVar strong / weak** constraint (L-BFGS on the B^{1/2}-preconditioned cost), with nominal, **Desroziers**
  (no truth) or **oracle** R, and the fair reference **`4dvar_tuned` / `4dvar_tuned_weak`**: b_scale, window
  (and q_scale) selected on the validation split — the 4DVar counterpart of 4DVarNet's training / early stopping —
  plus the Desroziers R,
* **incremental 4DVar** as run operationally (Gauss-Newton + preconditioned CG with TL / adjoint), at an
  operational-like budget (3 x 20) and at a **low budget** (1 x 5),
* the **hybrid 4DVarNet → 4DVar** (4DVarNet reconstruction as background of each window) and a small sweep,

with scores, the **error energy per Lyapunov subspace** and **paired block-bootstrap confidence intervals**
(against the tuned 4DVar, and against 4DVarNet for the hybrids).

Configuration: `contrib/qg2l/sweep_config.py` (shared with `python -m contrib.qg2l.run_sweep`, which runs the same
steps as GPU jobs, in parallel if wanted). Everything is cached by configuration hash and crash-safe: re-running
only computes what is missing.

In [ ]:
import os, sys, json, copy
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "main.py").exists():
    REPO = REPO.parent
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import xarray as xr
from IPython.display import Image, Video, display

from contrib.qg2l import pipeline, obs as O
from contrib.qg2l.simulate import PRESETS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 0. Configuration (`contrib/qg2l/sweep_config.py`)

In [ ]:
from contrib.qg2l import sweep_config, run_sweep

SIZE = "small"   # "small": all scenarios | "full": paper resolution (64x64, 1000 km), baseline + realistic
C = sweep_config.config(SIZE)
ROOT, RUN, METHOD_ORDER = C["ROOT"], C["RUN"], C["METHOD_ORDER"]
SEEDS, SEED_SCENARIOS, TUNE = C["SEEDS"], C["SEED_SCENARIOS"], C["TUNE"]
# C["RUN"] = ["baseline", "realistic"]   # e.g. restrict the scenarios to run / show
print("run:", RUN, "| seeds", SEEDS, "for", SEED_SCENARIOS, "| tuning grid", TUNE)
pd.DataFrame(C["METHODS"]).set_index("name").fillna("")

## 1. Scenarios

| scenario | what changes | assumption broken |
|---|---|---|
| `baseline` | nothing | — |
| `obs_hr` | obs sample the block-averaged **high-resolution** SSH | no representativeness error |
| `obs_signal` | + an **unresolved SSH signal** (internal-tide-like, 1 cm, 40–120 km, 1-day decorrelation) in the obs only | obs = model-resolvable truth + white noise |
| `swot_roll` | SWOT **roll-like correlated error** (1 cm / 100 km), R still diagonal in 4DVar | R known and diagonal |
| `param_error` | truth with **different physics** (bottom drag ×1.5, β ×0.8) | perfect model equations |
| `forcing` | truth with an **unknown stochastic forcing** (2 mm/day, 150–400 km) | perfect model forcing |
| `B_free_run` | 4DVar **B from a free run of the DA model** instead of the truth | oracle B |
| `nadir_only` | no SWOT | (observability: larger λ₁·τ_obs) |
| `realistic` | everything above together (with SWOT) | all |

In [ ]:
# scenario definitions (see sweep_config.py)
{n: s for n, s in C["SCENARIOS"].items() if n in RUN}

## 2. Run

**Following progress / running in parallel.** The cell below runs everything sequentially in the kernel. The same
steps can be run as separate GPU jobs (the notebook then only reads the cached results), e.g.

```bash
python -m contrib.qg2l.run_sweep --size small --scenario realistic                # all steps of one scenario
python -m contrib.qg2l.run_sweep --size small --scenario baseline --only seeds    # extra 4DVarNet seeds
python -m contrib.qg2l.run_sweep --size small --scenario param_error --only tuning
python -m contrib.qg2l.pipeline data/qg2l_scenarios_small                         # status (any time)
```

Do not run two jobs touching the same dataset at the same time (baseline and B_free_run share theirs), except
`--only seeds` together with `--only tuning`.

**After a crash**: restart and re-run; finished steps are reused, unfinished ones resume (4DVarNet from
`last.ckpt`, 4DVar from its last window, Lyapunov sweep from its last 10-day checkpoint).

**Cost of this round (small size, on top of the previous run)**: 4 extra 4DVarNet trainings (2 seeds x baseline,
realistic), 12 validation 4DVar runs per scenario for the tuning, 2 tuned 4DVars per scenario, and the
diagnostics (re-run because methods were added).

In [ ]:
RESULTS = {}
for name in RUN:
    print(f"\n========== {name} ==========")
    try:
        RESULTS[name] = run_sweep.run_one(C, name, device=DEVICE)
    except Exception as e:  # keep going with the other scenarios
        import traceback; traceback.print_exc()
        print(f"!!! scenario {name} failed: {e}")

In [ ]:
pipeline.progress(ROOT)   # state of datasets / trainings / scenarios (also after a restart)

## 3. Observation statistics and dynamical regime per scenario

In [ ]:
pd.DataFrame({n: {k: r["obs_summary"][k] for k in ["lambda1_per_day_LRmodel", "median_revisit_days",
                                                   "lambda1_x_revisit", "ssh_std_m", "obs_minus_truth_std_m"]}
              for n, r in RESULTS.items()}).T.round(4)

## 4. Scores (SSH and deep layer)

In [ ]:
scores = pd.concat({n: pd.DataFrame(r["scores"]).T for n, r in RESULTS.items()}, names=["scenario", "method"])
scores.round(4)

In [ ]:
# number of model integrations (nonlinear / tangent linear / adjoint) over the whole test run
pd.concat({n: pd.DataFrame(r["cost"]).T for n, r in RESULTS.items() if r.get("cost")}, names=["scenario", "method"])

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8), constrained_layout=True)
for a, key in zip(ax, ["rmse_ssh", "lx_ssh_km", "rmse_layer2", "lx_layer2_km"]):
    tab = scores[key].unstack("method").reindex(columns=[m for m in METHOD_ORDER if m in scores.index.get_level_values(1)])
    tab.loc[list(RESULTS)].plot.bar(ax=a, legend=a is ax[0], rot=30)
    a.set_title(key)

## 5. Error energy per Lyapunov subspace

In [ ]:
SUBS = ["blv_unstable_abs", "blv_neutral_abs", "blv_stable_resolved_abs", "residual_abs"]
rows = []
for n, r in RESULTS.items():
    if r["lyap"] is None:
        continue
    for m, v in r["lyap"]["methods"].items():
        rows.append(dict(scenario=n, method=m, n_unstable=r["lyap"]["n_unstable"],
                         total=v["mean_error_energy"], **{s: v["mean_" + s] for s in SUBS},
                         angle_us_deg=v.get("mean_angle_us_deg", np.nan)))
lyap_tab = pd.DataFrame(rows).set_index(["scenario", "method"])
lyap_tab

In [ ]:
fig, ax = plt.subplots(1, len(SUBS) + 1, figsize=(4.2 * (len(SUBS) + 1), 4), constrained_layout=True, sharey=False)
for a, key in zip(ax, ["total"] + SUBS):
    tab = lyap_tab[key].unstack("method")
    tab = tab.reindex(columns=[m for m in METHOD_ORDER if m in tab.columns]).loc[[n for n in RESULTS if n in tab.index]]
    tab.plot(ax=a, marker="o", logy=True, legend=a is ax[0], rot=30)
    a.set_title(key.replace("_abs", "").replace("blv_", ""))
    a.set_xticks(range(len(tab))); a.set_xticklabels(tab.index, rotation=30)
fig.suptitle("mean error energy per subspace (energy norm), by scenario");

In [ ]:
# relative gain of 4DVarNet over the tuned 4DVar (fallback: strong 4DVar) per subspace: > 0 means 4DVarNet is better
ref_m = "4dvar_tuned" if "4dvar_tuned" in lyap_tab.index.get_level_values(1) else "4dvar_strong"
gain = (1 - lyap_tab.xs("4dvarnet", level="method")[["total"] + SUBS]
        / lyap_tab.xs(ref_m, level="method")[["total"] + SUBS])
print("reference:", ref_m)
gain.round(3)

## 5b. Observation-error statistics used by the 4DVars

`nominal` = instrumental noise only; `desroziers` = per-satellite std estimated from innovation statistics on the
validation split (2 fixed-point iterations, no truth); `oracle` = actual std of obs − truth. With unresolved signal
in the observations (`obs_signal`, `realistic`) the nominal R is badly misspecified: compare `4dvar_strong`,
`4dvar_R_desroziers` and `4dvar_R_oracle` before concluding on these scenarios.

In [ ]:
import xarray as xr

rows = []
for n, r in RESULTS.items():
    obs_std = r["obs_summary"].get("obs_minus_truth_std_m")
    for m in ["4dvar_R_desroziers", "4dvar_R_oracle", "4dvar_tuned", "4dvar_tuned_weak", "hybrid_w3_Rdes"]:
        if m in r["recs"]:
            info = json.loads(xr.open_dataset(r["recs"][m], decode_times=False).attrs["config"]).get("r_info", {})
            est = info.get("desroziers", [None])[-1] if "desroziers" in info else info.get("oracle")
            rows.append(dict(scenario=n, method=m, obs_minus_truth_std=obs_std,
                             **{f"sigma_sat{k}": v for k, v in (est or {}).items()}))
pd.DataFrame(rows).set_index(["scenario", "method"]).round(4)

## 5c. Confidence intervals (paired moving-block bootstrap over test days)

Ratio = E_ref / E_method per subspace (> 1: the method has less error than the reference), 90 % intervals,
blocks of 10 days (~ Lyapunov time), same blocks for all methods (paired). `p_better` = bootstrap probability that
the ratio is > 1.

In [ ]:
from contrib.qg2l.stats import subspace_ci_from_file


def ci_table(ref, methods, scen=None):
    rows = []
    for n, r in RESULTS.items():
        f = r["diag_dir"] / "lyapunov_projection.nc"
        if f.exists() and (scen is None or n in scen):
            avail = json.loads(xr.open_dataset(f, decode_times=False).attrs["methods"])
            if ref in avail:
                for row in subspace_ci_from_file(f, ref=ref, block=10, n_boot=2000, level=0.9):
                    rows.append(dict(scenario=n, **row))
    df = pd.DataFrame(rows)
    if df.empty:
        return df, df
    tab = df[df.method.isin(methods)].copy()
    tab["ratio [90% CI]"] = tab.apply(lambda r: f"{r.ratio:.2f} [{r.ratio_lo:.2f}, {r.ratio_hi:.2f}]", axis=1)
    return df, tab.pivot_table(index=["scenario", "method"], columns="subspace", values="ratio [90% CI]",
                               aggfunc="first")


REF = "4dvar_tuned"
CI_METHODS = ["4dvarnet", "4dvarnet_s1", "4dvarnet_s2", "4dvar_strong", "4dvar_tuned_weak", "4dvar_R_desroziers",
              "hybrid_w3", "hybrid_w3_Rdes"]
ci, ci_tab = ci_table(REF, CI_METHODS)
ci_tab

In [ ]:
subs = ["unstable", "neutral", "stable_resolved", "residual", "total"]
show = [m for m in ["4dvarnet", "4dvar_strong", "4dvar_tuned_weak", "4dvar_R_desroziers", "hybrid_w3_Rdes"]
        if m in set(ci.method)]
fig, axes = plt.subplots(1, len(subs), figsize=(4 * len(subs), 4), sharey=True, constrained_layout=True)
scen = [n for n in RESULTS if n in set(ci.scenario)]
for ax, sub in zip(axes, subs):
    for j, m in enumerate(show):
        d = ci[(ci.subspace == sub) & (ci.method == m)].set_index("scenario").reindex(scen)
        x = np.arange(len(scen)) + (j - len(show) / 2) * 0.12
        ax.errorbar(x, d.ratio, yerr=[d.ratio - d.ratio_lo, d.ratio_hi - d.ratio], fmt="o", ms=4, capsize=2,
                    label=m)
    ax.axhline(1, c="gray", lw=0.8)
    ax.set_yscale("log"); ax.set_title(sub)
    ax.set_xticks(range(len(scen))); ax.set_xticklabels(scen, rotation=30)
axes[0].set_ylabel(f"E[{REF}] / E[method]  (> 1: better than {REF})")
axes[0].legend(fontsize=8);

**Hybrids against 4DVarNet** (paired, same blocks): ratio = E[4dvarnet] / E[method] > 1 means the hybrid is better
than 4DVarNet alone in that subspace.

In [ ]:
_, ci_hyb = ci_table("4dvarnet", ["hybrid", "hybrid_w3", "hybrid_b01", "hybrid_weak", "hybrid_w3_Rdes", "4dvar_tuned"])
ci_hyb

## 5d. 4DVar tuning on the validation split

Grid b_scale x window (strong constraint, oracle R, 20 L-BFGS iterations), then q_scale for the weak constraint at the
best (b_scale, window); objective = MSE of both layers on the validation period. The selected values feed
`4dvar_tuned` / `4dvar_tuned_weak` (with the Desroziers R).

In [ ]:
tun = []
for n, r in RESULTS.items():
    if r.get("tuning"):
        for row in r["tuning"]["table"]:
            tun.append(dict(scenario=n, **row))
tun = pd.DataFrame(tun)
display(pd.DataFrame({n: {**r["tuning"]["best_strong"], **{"q_scale (weak)": (r["tuning"]["best_weak"] or {}).get("q_scale")}}
                      for n, r in RESULTS.items() if r.get("tuning")}).T)
strong = tun[tun["mode"] == "strong"] if not tun.empty else tun
if not strong.empty:
    scen_t = list(dict.fromkeys(strong.scenario))
    fig, axes = plt.subplots(1, len(scen_t), figsize=(3.6 * len(scen_t), 3.2), constrained_layout=True, squeeze=False)
    for ax, n in zip(axes[0], scen_t):
        piv = strong[strong.scenario == n].pivot_table(index="b_scale", columns="window_days", values="objective")
        im = ax.imshow(np.sqrt(piv.values) * 100, origin="lower", cmap="viridis")
        ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns)
        ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
        ax.set_xlabel("window [days]"); ax.set_ylabel("b_scale"); ax.set_title(n)
        for (a, b), v in np.ndenumerate(np.sqrt(piv.values) * 100):
            ax.text(b, a, f"{v:.2f}", ha="center", va="center", color="w", fontsize=8)
    fig.suptitle("validation RMSE of the 4DVar state (both layers, cm)")

## 5e. 4DVarNet seed-to-seed variability

Three trainings (seeds 333, 1, 2) on the same data in the seed scenarios: spread of the scores and of the error energy
per subspace, and the ratio to the tuned 4DVar for every seed. If the seed spread is small compared with the gap to
4DVar, the conclusions do not depend on one lucky training.

In [ ]:
seed_names = ["4dvarnet"] + [f"4dvarnet_s{s}" for s in SEEDS[1:]]
rows = []
for n in [s for s in SEED_SCENARIOS if s in RESULTS]:
    for m in seed_names:
        if m in RESULTS[n]["scores"]:
            rows.append(dict(scenario=n, method=m, **RESULTS[n]["scores"][m],
                             **{k: RESULTS[n]["lyap"]["methods"][m]["mean_" + k] for k in SUBS},
                             total=RESULTS[n]["lyap"]["methods"][m]["mean_error_energy"]))
seed_tab = pd.DataFrame(rows)
if not seed_tab.empty:
    cols = ["rmse_ssh", "rmse_layer2", "lx_ssh_km", "total"] + SUBS
    summ = seed_tab.groupby("scenario")[cols].agg(["mean", "std"])
    summ.columns = [f"{a} {b}" for a, b in summ.columns]
    display(summ.T.round(6))
    if not ci.empty:
        display(ci[ci.method.isin(seed_names) & ci.scenario.isin(SEED_SCENARIOS)]
                .pivot_table(index=["scenario", "method"], columns="subspace", values="ratio").round(2))

## 6. Detailed diagnostic of one scenario

In [ ]:
SHOW = "realistic" if "realistic" in RESULTS else list(RESULTS)[0]
d = RESULTS[SHOW]["diag_dir"]
for f in ["lyapunov_spectrum_error_distribution.png", "error_energy_by_subspace.png", "subspace_summary.png",
          "clv_and_error_maps.png"]:
    if (d / f).exists():
        display(Image(filename=str(d / f)))
anim = d / "reconstructions.mp4"
if anim.exists():
    display(Video(str(anim), embed=True, width=1100))

In [ ]:
# 4DVarNet training curves for that scenario
hist = pipeline.training_curves(RESULTS[SHOW]["recs"]["4dvarnet"])
if not hist.empty:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
    hist[[c for c in ["train_loss", "val_loss"] if c in hist]].dropna(how="all").plot(ax=ax[0], logy=True)
    hist[[c for c in ["train_mse", "val_mse"] if c in hist]].dropna(how="all").plot(ax=ax[1], logy=True)
    ax[0].set_title("loss"); ax[1].set_title("MSE")

## 7. Reading the results

* **unstable** column: controlled by observability + optimisation. A dynamically consistent estimate (4DVar) lets its
  error grow along these directions within each window (saw-tooth in `error_energy_by_subspace.png`); the learned
  smoother does not have to be a model trajectory.
* **stable / residual** columns: in the baseline, 4DVar gets them almost for free from the exact dynamics. Model
  error (`param_error`, `forcing`) and unresolved signal in the obs (`obs_signal`) should erode this advantage;
  if it persists, the learned prior is the bottleneck (try a richer prior / more data).
* **4dvar_incr vs 4dvar_strong**: operational incremental solver vs (near) converged nonlinear minimisation.
  **4dvar_incr_budget**: what is left of a properly preconditioned 4DVar at a few TL/AD integrations per window
  (compare with the cost of a 4DVarNet forward pass in the cost table).
* **hybrid**: if it beats both, the two error structures are complementary (4DVarNet fixes the unstable part, the
  short 4DVar re-imposes dynamical consistency on the stable part).
* **4dvar_tuned** is the fair reference (hyper-parameters selected on validation, R from Desroziers): conclusions on
  4DVarNet vs 4DVar should be read against it, with the bootstrap intervals and the seed spread.
* **full size** (`SIZE = "full"`): re-run baseline and realistic at 64x64 / 1000 km to confirm at the paper
  resolution (`python -m contrib.qg2l.run_sweep --size full --scenario baseline realistic`).